<a href="https://colab.research.google.com/github/gabraxas/LVs-and-Policy/blob/main/week05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 · 항법, 유도, 제어
### Navigation, Guidance and Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabraxas/LVs-and-Policy/blob/main/lecture/week05/week05.ipynb)

> **우주수송정책과 발사체 기술** — Week 05
> 참고: Rho, Woong Rae, "Space Launch Vehicle Course, Navigation and Guidance System," 2024

> 이 노트북은 GitHub에 저장되고 Colab에서 실행됩니다. 위 배지를 눌러 Colab에서 열거나, 아래 첫 코드 셀부터 순서대로 실행하세요.

In [ ]:
# ▶ 실행 전 준비 — 이 셀을 먼저 실행하세요 (약 10~20초 소요)
!pip install -q ipywidgets
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/_shared/course_interactive.py",
    "course_interactive.py")

from course_interactive import *
setup_korean_font()
print("준비 완료 — 아래 셀들을 순서대로 실행하며 강의를 진행하세요.")


## 학습 목표

- [ ] **우주발사체의 항법을 이해함** —
- [ ] **우주발사체의 유도에 대해서 이해함** —

> **핵심 질문**: 우주발사체는 어떻게 길을 찾아 원하는 목적지까지 도착하는가?

## 교재 챕터4 안에서 이 강의의 위치

- Greenberg, Ch.4 "Space Operations" — 위성운용 경제성 전반(시뮬레이션 모델링, 발사체 선정, 궤도상 정비, 사용료 정책, 궤도잔해 등)을 폭넓게 다룸
- 본 강의는 이 중 재사용발사체(RLV)·차세대 수송 아키텍처의 경제성 비교와 직접 관련된 **§VI**에 집중

| 절 | 내용 |
|---|---|
| I | 시뮬레이션 모델링 |
| II~V | 발사체 선정 · 사용자 요금 |
| **VI** | **위험기반 아키텍처 비교 (본 강의)** |
| VII~IX | 정부임무 발사체 선정 · 비용위험 |
| X~XII | 회복탄력성 · 궤도상 정비 · 궤도잔해 |

## 출발점 — 성능 · 비용 · 일정의 세 자유도

- 신규 수송 아키텍처 개발에는 성능(performance)·비용(cost)·일정(schedule) 세 변수가 얽혀 있음
- 이 셋을 동시에 모두 확정할 수는 없음 — 하나 이상은 반드시 불확실성 변수로 남음

| 변수 | 취급 방식 |
|---|---|
| 성능 | 궤도 투입 질량 등 달성 능력 — **불확실 변수**로 취급 |
| 비용 | 개발·운용 비발생/반복 비용 — **불확실 변수**로 취급 |
| 일정 | 본 방법론에서는 **고정**으로 가정(단순화) |

> **단순화 가정**: 모든 일정은 알려져 있다고 가정하고, 성능 또는 비용(혹은 둘 다)에 개발비용 위험·성능 위험을 명시적으로 부여한다.

## 왜 기댓값만으로는 부족한가 — (m, σ) 프레임

- 각 아키텍처는 생애주기비용 현재가치의 **기대값 m**과 **표준편차 σ**로 이루어진 한 쌍의 지표로 특성화됨
- σ는 '위험'의 정량적 대리지표
- 두 아키텍처의 기대비용(m)이 동일해도, 분포가 넓게 퍼진 쪽은 예상외 초과비용 가능성이 큼
- 정책 결정자는 '기대비용이 낮은 대안'과 '위험이 낮은 대안' 사이의 상충을 명시적으로 인식해야 함
- 이 관계를 다루기 위해 Greenberg는 **몬테카를로 시뮬레이션**으로 (m, σ)를 계산하는 절차를 제시

## 비교 대상 수송 아키텍처

| 구분 | 기존/개량 ELV·우주왕복선 | SSTO (단일단 궤도직접도달) | TSTO (2단 궤도도달) |
|---|---|---|---|
| 개발 성격 | 이미 운용 중, 성능·비용 불확실성 낮음 | 신규 단일단 완전재사용 개념, 불확실성 최대 | 1단·2단 개별 개발, 단간 성능 연계 필요 |
| 필요 데이터 | 상대적으로 소수의 입력변수 | 설계점·최소허용성능, 비발생비용 범위 등 7종+ | 1단·2단 각각의 성능·비용 범위 및 연계함수 등 11종+ |
| 핵심 위험 요인 | 운용 실적 기반, 위험 낮음 | 목표 성능 미달 시 비발생비용 급증 | 1단 성능이 2단 성능분포에 영향(연쇄 위험) |

## SSTO 분석 — 성능 불확실성 → 비용분포

1. 달성 궤도투입질량(capability)의 사전확률분포에서 무작위 표본을 추출
2. 표본이 **최소허용성능 미만**이면 → 최소성능은 달성된 것으로 간주, 비발생비용을 최댓값(Pa)으로 고정
3. 표본이 **설계점(design point)을 초과**하면 → 설계점으로 절단, 비발생비용을 최솟값(Pb)으로 고정
4. 표본이 두 값 사이면 → 비발생비용의 사전확률분포에서 대응 표본을 추출
5. 이 과정을 대량 반복(Monte Carlo) → 비발생비용의 **사후확률분포** 도출

> 달성 성능(capability)이 최소허용성능~설계점~최대가능성능 구간을 지나며 비발생비용이 결정되는 절차가 SSTO 방법론의 핵심

## 탑재체 설계여유(margin)로 전이되는 발사체 위험

- 발사체가 설계점 성능에 미달하면 탑재체(payload)의 설계여유가 줄어듦
- 설계여유 축소는 탑재체 비발생·반복 비용의 증가로 이어짐 — **발사체의 성능 위험이 탑재체 비용 위험으로 전이**되는 경로

$$
\text{P/L Margin} = [\text{사전 설계여유}] + \frac{\text{달성 궤도투입질량}}{\text{설계점 궤도투입질량}} - 1.0
$$

발사체 성능 미달 → 탑재체 설계여유 축소 → 탑재체 비용 증가(민감도계수)

> **모델링 방식**: 설계여유 감소분에 대해 2차 다항식 형태의 민감도계수를 적용하여 비발생비용 범위(최소/최대/최빈값)를 재산정한다.

## 연간비용에서 생애주기비용 현재가치까지

**연간비용** (년 I, 시뮬레이션 회차 R)

$$
\begin{aligned}
\text{Annual Cost}(I,R) = &\ \text{수송시스템 비발생비용} + \text{함대 자본투자} + \text{발사장 자본투자} \\
&+ \left[\text{RLV 단위질량당 수송비} + K\cdot\text{탑재체 비발생비용/질량} + \text{탑재체 반복비용/질량}\right] \times \text{RLV 수송질량} \\
&+ \left[\text{ELV 단위질량당 수송비} + K\cdot\text{탑재체 비발생비용/질량} + \text{탑재체 반복비용/질량}\right] \times \text{ELV 수송질량}
\end{aligned}
$$

**생애주기비용 현재가치(PVLCC)**

$$
\text{PVLCC}(R) = \sum_{I} \frac{\text{Annual Cost}(I,R)}{(1+d)^{I}}
$$

($d$ = 할인율, $I$ = 연차, $R$ = 몬테카를로 반복 회차)

**기대값(m)과 표준편차(σ)** — MAXR회 반복

$$
m = \frac{\sum_R \text{PVLCC}(R)}{\text{MAXR}} \qquad
\sigma = \sqrt{\frac{\sum_R \text{PVLCC}(R)^2}{\text{MAXR}} - m^2}
$$

- 각 아키텍처마다 이 절차를 대량 반복 → 대안별로 비교 가능한 (m, σ) 한 쌍 산출

**[인터랙티브] 몬테카를로 위험비교 탐색기** — 위 절차를 직접 실행, 성능 불확실성·비용민감도가 (m, σ)를 어떻게 바꾸는지 확인

In [ ]:
monte_carlo_risk_explorer()

## TSTO — 1단·2단 성능의 연쇄적 불확실성

- TSTO(2단형)는 2단 로켓의 성능·비용이 1단 로켓이 실제로 달성한 성능에 **조건부로 의존** — SSTO 분석과의 핵심 차이

1. 1단 성능 표본추출: $S1CAP(R)$
2. 1단 비발생비용 결정: $S1NRC(R)$
3. 2단 성능 보정: $S2CAP(R) = f(S1CAP)$
4. 2단 비발생비용 결정: $S2NRC(R)$

$S2CAP(R)$ 보정식(설계점 기준 % 편차):

$$
S2CAP(R) = S2S1A \times [S1CAP(R) - 100] + K_1 \times S2S1B \times [S1CAP(R) - 100]^2 + S2CAP(R)
$$

($S1CAP > 100$일 때 $K_1 = 1.0$, $S1CAP < 100$일 때 $K_1 = -1.0$ — 설계점 대비 초과/미달 방향에 따라 비대칭 반영)

- TSTO 분석은 SSTO 대비 최소 11종 이상의 핵심 입력데이터(양 단의 설계점·성능범위·비발생비용범위·연계함수 등) 요구 — 데이터 요구수준이 아키텍처 복잡도와 함께 증가

## STARS — 실행 도구로서의 몬테카를로 모델

- **STARS** (Space Transportation Architecture Risk System) — 앞선 방법론을 실제로 구현한 도구, 1990년대 NASA의 첨단 수송개념 비교에 사용
- Excel 스프레드시트 구조 내에서 완전히 동작하는 몬테카를로 시뮬레이션
- 133MHz급 PC에서 100회 시뮬레이션이 수 초 내 완료 — 당시 기준 신속한 반복 분석
- 상세 물리모형 대신 '불확실성 범위' 입력에 의존하는 고수준 추상화 모델
- SSTO·TSTO에 기존/경쟁 수송수단(시장점유율 변화 포함)까지 함께 고려 가능
- 사용자 친화적 메뉴 기반 입력화면과 수치·그래프 출력 제공

**STARS의 목적**: NASA 기술프로그램 투자결정을 지원하기 위해, 다수의 첨단 수송 아키텍처를 신속·반복적으로 비교하는 초기 스크리닝 도구

> 입력: 성능·비용 불확실성 범위 → 처리: 몬테카를로 무작위 표본추출 → 출력: 아키텍처별 (m, σ) 및 연차별 비용

## 사례연구 — 고도재사용 수송(HRST) · 마그레브 아키텍처

- STARS 활용 예시로 Greenberg는 초전도 자기부상(maglev) 캐터펄트 방식의 고도재사용 지구-궤도 수송개념을 제시
- **수치는 예시 목적일 뿐, 실제 설계 평가로 해석해서는 안 됨**

| 구성요소 | 설명 |
|---|---|
| 구조 지지체 | 산악 터널형 가속/감속 구간(수 마일 길이)의 견고한 구조 지지시스템 |
| 마그레브 가이드웨이 | 저밀도 고음속 기체(예: 헬륨)로 채운 가속관 + 감속 구간 |
| 전력저장 시스템 | 초전도 자기에너지 저장 등, 지역 전력망에서 충전 후 발사 시 방전 |
| 재사용 가속운반체 | 발사체를 지지·가속하고 궤도 진입 시점에 정밀 분리하는 캐리어 |
| 발사/이탈 시스템 | 터널 내부→외기환경으로의 능동제어 전이 구간 |
| 지상 통합시설 | 캐리어 스테이징·발사체 통합·정비·운영관제 센터 |

> **특징**: 극단적 가속도가 불필요, 탑재체(위성) 설계의 급격한 변경이나 초고빈도 발사가 없어도 경제성을 확보하도록 설계

## 아키텍처 연차별 비용 프로파일 (예시 재구성)

- STARS 산출물 형태를 단순화하여 재구성한 예시
- HRV(재사용) 도입 초기에는 비발생비용이 지배적이나, 운용이 안정화되며 반복비용이 비용구조를 주도

- **초기(2009~2011)**: 함대투자·1단/2단 비발생비용이 큰 비중을 차지
- **중반 이후**: ELV 병행운용 비중이 줄고 HRV 반복비용이 증가
- 표준편차(σ)는 매 연차·항목마다 별도로 산출되어 위험 프로파일 형성
- 탑재체 관련 비용을 0으로 두면 순수 수송 비용만 분리 관찰 가능

## 위험-기댓값 상충관계 — 최적 대안 프론티어

- 각 아키텍처를 (기대현재가치비용, 위험) 평면 위의 한 점으로 표시 → 어떤 대안도 '비용도 낮고 위험도 낮을' 수는 없다는 상충관계가 드러남
- 지배되지 않는 점들의 집합이 **'최적 대안 프론티어'**

- 동일 위험(σ)이면 기대비용이 낮은 대안이 우월
- 동일 기대비용이면 위험이 낮은 대안이 우월
- 프론티어 위 대안들 사이의 최종 선택은 의사결정자의 **위험선호(risk appetite)**에 달림

> 의사결정자는 프론티어 위에서 '비용 절감 vs 위험 증가'의 트레이드오프를 선택한다.

**[인터랙티브] (m, σ) 위험-기댓값 프론티어 탐색기** — ELV·SSTO·TSTO를 평면 위에 놓고 어떤 대안이 프론티어를 구성하는지 확인

In [ ]:
risk_frontier_explorer()

## 방법론의 한계와 확장 방향

- **공통 수요 시나리오 가정**: 아키텍처마다 비용(가격)이 다르면 수요도 달라질 수 있으나, 비교 편의를 위해 동일한 수요(연간 궤도투입 질량)를 가정
- **가격탄력성 미반영**: 비용 차이가 크지 않은 대안 간 비교에는 무리가 없으나, 비용 차이가 큰 대안 간에는 결과가 비용격차를 과장할 수 있음
- **확장 가능성**: STARS에 가격·수요 상호작용 모듈을 추가하면 한계를 완화할 수 있으나 모델 복잡도가 크게 증가
- **실무적 절충**: 완전한 가격-수요 모델 없이도, 유사한 비용대의 아키텍처를 비교하는 초기 스크리닝에는 현재 방법론으로 충분

## 정책적 시사점과 토론

> **정책메모 연계**: "재사용발사체 개발 투자 대비 해외 발사서비스 구매의 타당성 검토" 정책메모에 본 강의의 (m, σ) 프레임과 최적 대안 프론티어를 적용해보라. 기대비용뿐 아니라 위험을 함께 언급해야 설득력 있는 권고가 된다.

1. 정부는 왜 SSTO·TSTO처럼 미성숙한 기술에 R&D 투자를 정당화하기 위해 '위험'을 명시적으로 계량해야 하는가
2. 동일한 기대비용을 갖는 두 아키텍처 중 표준편차가 큰 쪽을 선택해야 하는 상황이 있다면 어떤 경우인가
3. 한국의 KSLV-III 재사용 발사체 개발에 이 프레임(m, σ, frontier)을 적용한다면 어떤 입력데이터가 가장 확보하기 어려울까

## 참고문헌

**[주교재]** Greenberg, J.S., "Space Operations," Chapter 4 in *Progress in Astronautics and Aeronautics*, Vol. 201: *Economic Principles Applied to Space Industry Decisions*, AIAA, Reston, VA — §VI "Risk-Based Approach for Comparing Advanced Transportation Architectures," pp. 243–271.

**보조 참고자료** (교재 원문 각주)
- Greenberg, J.S., "RLV Pricing Strategies," NASA, July 1997.
- Shaw, E.J., Taylor, D.T., Hamaker, J.W., "RLV Economics: Fiscal Evaluation of NASA's Reusable Launch Vehicle Effort," *Space Policy*, May 1997.
- Greenberg, J.S., "Insuring RLV Transportation Services," International Academy of Astronautics, Paper 98-IAA.1.2.02, 1998.

---
**다음 주(6주차) 예고**: 우주수송의 역사와 정책의 기원 — 냉전 우주경쟁과 발사체 개발사, 국가 위신·안보 동인